# PAIMANA Phase 2 model development

This notebook demonstrates the production workflow in `ml/phase2.py`. It does not duplicate the training pipeline. Run from a clean kernel with the repository root on `sys.path`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "ml" / "phase2.py").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ml.phase2 import (
    ARTIFACT_DIR,
    COMPLETED_PATH,
    FORBIDDEN_PREDICTORS,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    ONGOING_PATH,
    OUTPUT_PATH,
    _prepare_model_rows,
    audit_cleaned_data,
    build_snapshot_features,
    load_phase2_artifacts,
    predict_cost_overrun,
    predict_project_risk,
    predict_schedule_overrun,
    prepare_snapshot_source,
    run_temporal_integrity_check,
    score_ongoing_projects,
    train_phase2,
    write_leakage_audit,
)
from sklearn.model_selection import GroupKFold
print("ROOT", ROOT)
print("completed exists", COMPLETED_PATH.exists())

## Data loading and dataset integrity

In [ ]:
completed_raw = pd.read_csv(COMPLETED_PATH, low_memory=False)
ongoing_raw = pd.read_csv(ONGOING_PATH, low_memory=False)
completed, completed_integrity = prepare_snapshot_source(completed_raw)
ongoing, ongoing_integrity = prepare_snapshot_source(ongoing_raw)
print(completed_integrity)
print("unique completed projects", completed_integrity["unique_projects"])
print("chronological", completed_integrity["chronological_by_project"])
assert completed_integrity["chronological_by_project"]
assert completed["Project_ID"].notna().all()

## Feature engineering (Project_ID + Snapshot_Date, six-month lookback)

In [ ]:
sample_id = completed["Project_ID"].iloc[0]
sample = completed.loc[completed["Project_ID"].eq(sample_id)]
features = build_snapshot_features(sample)
print(features[["Project_ID", "Snapshot_Date"]].head())
print("feature count", len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES))
assert not features.duplicated(["Project_ID", "Snapshot_Date"]).any()

## Target creation and leakage audit

In [ ]:
audit = audit_cleaned_data()
write_leakage_audit()
print(audit["target_class_distribution_project_level"])
assert not set(NUMERIC_FEATURES + CATEGORICAL_FEATURES).intersection(FORBIDDEN_PREDICTORS)

## GroupKFold (project holdout)

In [ ]:
X, model_rows, outcomes, target_definitions, data_counts = _prepare_model_rows(completed, 10.0, 3.0)
groups = model_rows["Project_ID"].astype(str).to_numpy()
y = model_rows["Cost_Overrun_Flag"].astype(int).to_numpy()
for fold, (train_idx, val_idx) in enumerate(GroupKFold(n_splits=5).split(X, y, groups), start=1):
    train_ids = set(groups[train_idx].tolist())
    val_ids = set(groups[val_idx].tolist())
    assert train_ids.isdisjoint(val_ids)
    print(fold, "train projects", len(train_ids), "validation projects", len(val_ids))
print("target definitions", target_definitions["cost"]["class_counts"], target_definitions["schedule"]["class_counts"])
print(data_counts)

## Model training, evaluation, risk combination, thresholds, artifacts

If artifacts already exist, this cell loads them. Otherwise it runs the production trainer (GroupKFold, both models, nested risk combination, ongoing scoring).

In [ ]:
required = [
    "cost_model.pkl",
    "schedule_model.pkl",
    "risk_combination_model.pkl",
    "preprocessing.pkl",
    "feature_list.json",
    "target_definition.json",
    "risk_thresholds.json",
    "model_metrics.json",
    "model_metadata.json",
]
if all((ARTIFACT_DIR / name).exists() for name in required):
    print("Artifacts present; skipping retrain")
    artifacts = load_phase2_artifacts()
    print("loaded version", artifacts["metadata"]["model_version"])
else:
    result = train_phase2()
    print("trained cost ROC", result["model_metrics"]["cost_model"]["summary_mean_std"]["roc_auc"])
    artifacts = load_phase2_artifacts()

## Ongoing-project inference

In [ ]:
scored = score_ongoing_projects()
print(scored["report"]["projects_scored"], "projects",
      scored["report"]["rows_scored"], "rows",
      scored["report"]["failed_or_invalid_prediction_rows"], "failed")
one = ongoing.loc[ongoing["Project_ID"].eq(ongoing["Project_ID"].iloc[0])]
print(predict_project_risk(one))
print("cost", predict_cost_overrun(one), "schedule", predict_schedule_overrun(one))
assert OUTPUT_PATH.exists()
predictions = pd.read_csv(OUTPUT_PATH)
print(predictions.head())

## Temporal integrity

In [ ]:
temporal = run_temporal_integrity_check(ongoing, sample_projects=5)
print(temporal)
assert temporal["passed"]